# 6.1 Tasa de establecimiento de especies exóticas invasoras.

In [ ]:
import datetime
now = datetime.datetime.now()
print(f"Última actualización de este notebook: {now}")

Última actualización de este notebook: 2026-03-13 12:36:23.800538


El Indicador de Cabecera 6.1 evalúa la magnitud y velocidad de la amenaza que representan las invasiones biológicas para la biodiversidad. Su objetivo es medir la tasa anual a la cual nuevas Especies Exóticas Invasoras (EEI) logran establecerse en el país. El monitoreo de esta métrica es fundamental para dar seguimiento a la Meta 6 del Marco Mundial de Biodiversidad, orientada a reducir las tasas de introducción y establecimiento de especies invasoras y mitigar sus impactos.
Los metadatos oficiales del indicador se encuentran disponibles [aquí](https://www.gbf-indicators.org/metadata/headline/6-1).

## Metodología de Cálculo 
La métrica base del reporte se construyó utilizando el año documentado de primera detección (first record) en el territorio nacional. Para aislar el subconjunto de especies de interés, el equipo consultor siguió el flujo de trabajo propuesto por Seebens et al. (2020) y alineado con Pagad et al., (2022). Del total de especies introducidas, se seleccionaron únicamente aquellas explícitamente clasificadas como "Invasoras" y dañinas (marcador isInvasive = 1).
Durante el análisis, el equipo evaluó enfoques metodológicos alternativos, tales como el uso de ventanas temporales móviles (p. ej., promedios de cinco años) o la normalización por el número total de especies exóticas. Sin embargo, se decidió descartar estos ajustes estadísticos y reportar únicamente el conteo anual bruto de nuevos registros. Esta decisión metodológica se tomó para garantizar la transparencia, evitar introducir supuestos nacionales de suavizado temporal y asegurar la comparabilidad y agregación de los datos chilenos a escala global.

# Capa 1: Observación y recolección

## Fuentes de Datos Utilizadas
Los datos de especies fueron compilados a partir de la integración de dos repositorios científicos e internacionales de alto estándar:
- El Registro Global de Especies Introducidas e Invasoras para Chile (GRIIS – Chile), accesible a través de la plataforma GBIF (Global Biodiversity Information Facility).
- La Base de Datos de Primeros Registros de Especies Exóticas (Versión 3), publicada por Seebens et al. y disponible a través del repositorio Zenodo. Esta fuente fue clave para obtener los años precisos de detección.

In [4]:
# !pip install pandas plotly requests
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import requests


DATA_DIR = Path("./data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Descarga data desde GBIF

In [5]:
def download_data_from_gbif(url: str, output_dir: Path) -> Path:
    file_name = url.split("?r=")[-1] + ".zip"
    response = requests.get(url, stream=True)
    with open(output_dir / file_name, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024):
            f.write(chunk)

    return output_dir / file_name


def unzip_file(zip_file_path: Path, extract_folder: Path):
    with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
        zip_ref.extractall(extract_folder)


urls = [
    "https://cloud.gbif.org/griis/archive.do?r=juan-fernandez-islands-chile-griis",
    "https://cloud.gbif.org/griis/archive.do?r=chile-griis-gbif",
    "https://cloud.gbif.org/griis/archive.do?r=rapa-nui-chile-griis-gbif",
]

for url in urls:
    file_name = download_data_from_gbif(url, DATA_DIR)
    unzip_file(zip_file_path=file_name, extract_folder=DATA_DIR / str(file_name.stem))

## Descarga data desde Zenodo

In [6]:
def download_zenodo_dataset(url: str, output_dir: Path, file_name: str):
    response = requests.get(url, stream=True)
    with open(output_dir / file_name, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024):
            f.write(chunk)


url = "https://zenodo.org/records/10039630/files/GlobalAlienSpeciesFirstRecordDatabase_v3.1_freedata.xlsx?download=1"
download_zenodo_dataset(
    url=url,
    output_dir=DATA_DIR,
    file_name="GlobalAlienSpeciesFirstRecordDatabase_v3.1_freedata.xlsx",
)

In [7]:
df_alien_species = pd.read_excel(
    DATA_DIR / "GlobalAlienSpeciesFirstRecordDatabase_v3.1_freedata.xlsx",
    sheet_name="FirstRecords",
    dtype={
        "TaxonName": "category",
        "OrigName": "category",
        "scientificName": "category",
        "FirstRecord_orig": str,
        "Family": "category",
        "Order": "category",
    },
)

df_alien_species_chile = (
    df_alien_species[df_alien_species["Region"] == "Chile"]
    .reset_index(drop=True)
    .copy()
)

In [8]:
# df_alien_species_chile[df_alien_species_chile["FirstRecord"] >= 1970].hist()
fig = px.histogram(
    df_alien_species_chile,
    x="FirstRecord",
    title=f"Primer año de registro de especies invasoras entre {df_alien_species_chile['FirstRecord'].min()} y {df_alien_species_chile['FirstRecord'].max()}",
)
fig.show()

In [9]:
fig = px.histogram(
    df_alien_species_chile[df_alien_species_chile["FirstRecord"] >= 1970],
    x="FirstRecord",
    title=f"Primer año de registro de especies invasoras entre {df_alien_species_chile[df_alien_species_chile['FirstRecord'] >= 1970]['FirstRecord'].min()} y {df_alien_species_chile[df_alien_species_chile['FirstRecord'] >= 1970]['FirstRecord'].max()}",
    nbins=50,
)
fig.show()

In [10]:
#Data Alien Species Chile
df_alien_species_chile["FirstRecordDate"] = (
    df_alien_species_chile["FirstRecord"]
    .apply(lambda x: str(x) + "-01-01")
    .apply(np.datetime64)
)

In [11]:
# Data Chile
def get_data(dir: Path | str) -> pd.DataFrame:
    if isinstance(dir, str):
        dir = Path(dir)

    df_distribution = pd.read_csv(dir / "distribution.txt", sep="\t")
    df_distribution.set_index("id", inplace=True)

    df_taxon = pd.read_csv(dir / "taxon.txt", sep="\t")
    df_taxon.set_index("id", inplace=True)

    df_species_profile = pd.read_csv(dir / "speciesprofile.txt", sep="\t")
    df_species_profile.set_index("id", inplace=True)

    return pd.concat([df_taxon, df_distribution, df_species_profile], axis=1)


df_chile_griis = get_data(DATA_DIR / "chile-griis-gbif")
df_chile_griis["source"] = "chile-griis-gbif"

df_chile_juan_fernandez = get_data(DATA_DIR / "juan-fernandez-islands-chile-griis")
df_chile_juan_fernandez["source"] = "juan-fernandez-islands-chile-griis"

df_chile_rapa_nui = get_data(DATA_DIR / "rapa-nui-chile-griis-gbif")
df_chile_rapa_nui["source"] = "rapa-nui-chile-griis-gbif"

# Capa 2: Análisis y síntesis
## Resultados
De acuerdo con el cruce de las bases de datos internacionales, se logró identificar un universo inicial de 1.035 especies exóticas con información de primer registro documentado en el país. Al aplicar el filtro de impacto ecológico (isInvasive = 1), los resultados consolidados indican que Chile reporta a la fecha un total histórico acumulado de 287 Especies Exóticas Invasoras (EEI) que cuentan con un año de primera detección confirmado. El reporte anual enviado a CBD refleja esta señal de establecimiento de manera cronológica, proveyendo la base exacta requerida por el Indicador 6.1.


In [12]:
df_alien_species_chile["TaxonNameId"] = df_alien_species_chile["TaxonName"].str.lower()

In [13]:
df_griis = pd.concat([df_chile_griis, df_chile_juan_fernandez, df_chile_rapa_nui])
df_griis["id_name"]: str = None
for i, r in df_griis.iterrows():
    if isinstance(r["acceptedNameUsage"], str):
        # print(f'{" ".join(r["acceptedNameUsage"].split(" ")[:2])}')
        df_griis.at[i, "id_name"] = (
            f'{" ".join(r["acceptedNameUsage"].split(" ")[:2]).lower()}'
        )
    else:
        # print(f'{" ".join(r["scientificName"].split(" ")[:2])}')
        df_griis.at[i, "id_name"] = (
            f'{" ".join(r["scientificName"].split(" ")[:2]).lower()}'
        )

In [14]:
df_6_1_especies_invasoras = pd.merge(
    left=df_alien_species_chile[
        ["TaxonNameId", "TaxonName", "FirstRecordDate", "FirstRecord"]
    ],
    right=df_griis[["id_name", "isInvasive"]],
    how="outer",
    left_on="TaxonNameId",
    right_on="id_name",
).reset_index(drop=True)

df_6_1_especies_invasoras.drop_duplicates(inplace=True)

In [15]:
df_6_1_especies_invasoras.reset_index(drop=True, inplace=True)

In [16]:
griis = 0
joined = 0
zenodo = 0
for i, r in df_6_1_especies_invasoras.iterrows():
    if r["TaxonNameId"] == r["id_name"]:
        joined += 1
    elif isinstance(r["TaxonNameId"], float) and r["id_name"]:
        griis += 1
    elif r["TaxonNameId"] and isinstance(r["id_name"], float):
        zenodo += 1

print(f"Solo GRIIS: {griis} | Joined: {joined} | Solo introducidas: {zenodo}")

Solo GRIIS: 301 | Joined: 684 | Solo introducidas: 365


In [17]:
df_6_1_especies_invasoras["isInvasiveNum"] = df_6_1_especies_invasoras[
    "isInvasive"
].apply(lambda x: 1 if x == "invasive" else 0)

In [18]:
N_INVASIVE_SPECIES = df_6_1_especies_invasoras.isInvasiveNum.sum()

In [19]:
df_6_1_especies_invasoras_v2 = (
    df_6_1_especies_invasoras[df_6_1_especies_invasoras["FirstRecord"] >= 1970]
    .groupby("FirstRecord")
    .agg({"TaxonName": "count", "isInvasiveNum": "sum"})
    .reset_index()
)
df_6_1_especies_invasoras_v2["rate"] = (
    df_6_1_especies_invasoras_v2["isInvasiveNum"]
    / df_6_1_especies_invasoras_v2["TaxonName"]
    * 100
)

df_6_1_especies_invasoras_v2["rate2"] = (
    df_6_1_especies_invasoras_v2["isInvasiveNum"] / N_INVASIVE_SPECIES * 100
)
df_6_1_especies_invasoras_v2["rate2"] = df_6_1_especies_invasoras_v2["rate2"].round(2)

# Capa 3: Reporte Digital

In [20]:
y_data = (
    df_6_1_especies_invasoras_v2[["TaxonName", "isInvasiveNum"]]
    .groupby(df_6_1_especies_invasoras_v2.index // 5)
    .sum()
)
y_data["rate"] = y_data["isInvasiveNum"] / y_data["TaxonName"] * 100
y_data["rate"] = y_data["rate"].round(2)
y_data["rate2"] = y_data["isInvasiveNum"] / N_INVASIVE_SPECIES * 100
y_data["rate2"] = y_data["rate2"].round(2)

x_labels = (
    df_6_1_especies_invasoras_v2["FirstRecord"]
    .groupby(df_6_1_especies_invasoras_v2.index // 5)
    .first()
)
df_6_1_especies_invasoras_v3 = y_data.join(x_labels)
# df_6_1_especies_invasoras_v2.rolling(5).sum()

In [21]:
fig = px.line(
    df_6_1_especies_invasoras_v2, x="FirstRecord", y="rate2", title="Tasa de invasión"
)
fig.show()

In [22]:
fig = px.line(
    df_6_1_especies_invasoras_v3,
    x="FirstRecord",
    y="rate2",
    title=f"Tasa de establecimiento de especies invasoras entre {df_6_1_especies_invasoras_v3['FirstRecord'].min():.0f} y {df_6_1_especies_invasoras_v3['FirstRecord'].max():.0f}",
)
fig.show()

In [23]:
fig = px.bar(
    df_6_1_especies_invasoras_v3,
    x="FirstRecord",
    y="rate2",
    title=f"Tasa de especies invasoras entre {df_6_1_especies_invasoras_v3['FirstRecord'].min():.0f} y {df_6_1_especies_invasoras_v2['FirstRecord'].max():.0f}",
    text_auto=True,
)
fig.show()

In [24]:
fig = px.bar(
    df_6_1_especies_invasoras_v3,
    x="FirstRecord",
    y="rate",
    title=f"Tasa de especies invasoras entre {df_6_1_especies_invasoras_v3['FirstRecord'].min():.0f} y {df_6_1_especies_invasoras_v2['FirstRecord'].max():.0f}",
    text_auto=True,
)
fig.show()

In [25]:
import pandas as pd

df = df_6_1_especies_invasoras_v3.copy()

# Asegurar que FirstRecord sea "año" (int)
# - Si ya es numérico (ej. 2019), esto lo deja igual.
# - Si es datetime/string tipo "2019-01-01", lo convierte a año.
if not pd.api.types.is_integer_dtype(df["FirstRecord"]) and not pd.api.types.is_float_dtype(df["FirstRecord"]):
    df["FirstRecord"] = pd.to_datetime(df["FirstRecord"], errors="coerce").dt.year

df["FirstRecord"] = df["FirstRecord"].astype("Int64")  # permite NA si existieran

# Últimos 5 años (según el máximo año presente)
max_year = int(df["FirstRecord"].max())
min_year = max_year - 4

tabla_ult_5 = (
    df.loc[df["FirstRecord"].between(min_year, max_year), ["FirstRecord", "rate2", "rate", "isInvasiveNum", "TaxonName"]]
      .sort_values("FirstRecord")
      .rename(columns={
          "FirstRecord": "Año",
          "rate2": "Tasa (rate2) %",      # invasoras / N_INVASIVE_SPECIES * 100
          "rate": "Tasa (rate) %",        # invasoras / total * 100
          "isInvasiveNum": "N invasoras",
          "TaxonName": "N total"
      })
      .reset_index(drop=True)
)

tabla_ult_5

,Año,Tasa (rate2) %,Tasa (rate) %,N invasoras,N total
0,2015,1.05,15.79,3,19


In [26]:
df_6_1_especies_invasoras_v3["FirstRecord"].min(), df_6_1_especies_invasoras_v3["FirstRecord"].max()

(np.float64(1970.0), np.float64(2015.0))

In [27]:
tabla_completa = (
    df_6_1_especies_invasoras_v2
    .sort_values("FirstRecord")
    .rename(columns={
        "FirstRecord": "Año",
        "rate": "Tasa (%)",
        "rate2": "Tasa respecto total invasoras (%)",
        "isInvasiveNum": "N invasoras",
        "TaxonName": "N total"
    })
    .reset_index(drop=True)
)

tabla_completa

,Año,N total,N invasoras,Tasa (%),Tasa respecto total invasoras (%)
0,1970.0,5,0,0.000000,0.00
1,1971.0,10,0,0.000000,0.00
2,1972.0,5,1,20.000000,0.35
3,1973.0,3,2,66.666667,0.70
4,1974.0,8,0,0.000000,0.00
5,1975.0,4,1,25.000000,0.35
6,1976.0,5,1,20.000000,0.35
7,1977.0,1,0,0.000000,0.00
8,1978.0,4,1,25.000000,0.35
9,1979.0,8,0,0.000000,0.00


In [28]:
# Contar nuevas invasoras por año
nuevas_por_año = (
    df_6_1_especies_invasoras_v2
    .groupby("FirstRecord")
    .size()
    .reset_index(name="new_invasive_species")
    .sort_values("FirstRecord")
)

In [32]:
tabla_completa = (
    df_6_1_especies_invasoras_v2
    .sort_values("FirstRecord")
    [["FirstRecord", "TaxonName", "isInvasiveNum", "rate"]]
    .rename(columns={
        "FirstRecord": "año",
        "TaxonName": "N de exóticas",
        "isInvasiveNum": "N de invasoras",
        "rate": "tasa"
    })
    .reset_index(drop=True)
)

tabla_completa.tail(7)


,año,N de exóticas,N de invasoras,tasa
43,2013.0,3,0,0.000000
44,2014.0,5,0,0.000000
45,2015.0,1,0,0.000000
46,2016.0,1,0,0.000000
47,2017.0,3,0,0.000000
48,2018.0,13,3,23.076923
49,2019.0,1,0,0.000000


In [33]:
df_6_1_especies_invasoras.head(20)
#vamos a calcular el número total de especies invasoras que son aquellas que en la columna isInvasiveNum tienen el valor 1
total_especies_invasoras = df_6_1_especies_invasoras[df_6_1_especies_invasoras["isInvasiveNum"] == 1].shape[0]
total_especies_invasoras
print("Número de especies clasificadas como exóticas para Chile:", total_especies_invasoras)

Número de especies clasificadas como exóticas para Chile: 287


## Observaciones
El enfoque adoptado para este ciclo de reporte prioriza la fidelidad del 'dato crudo' (año de primer registro) por sobre el modelado estadístico. Esta estrategia metodológica alinea a Chile con las prácticas de estandarización exigidas por el Marco Mundial de Biodiversidad, facilitando los futuros análisis inter-países. Al presentar la cifra acumulada de 287 especies, el país establece una línea base comparable internacionalmente.

## Brechas
Se reconoce que la dependencia en los datos de 'primer registro' conlleva limitaciones analíticas que deben ser abordadas en el mediano plazo:
- Variabilidad en la probabilidad de detección: Los conteos anuales observados están fuertemente influenciados por el esfuerzo de muestreo histórico. Un aumento ('peak') en la cantidad de EEI detectadas en una década reciente puede reflejar un mayor financiamiento y esfuerzo en la investigación científica nacional, más que un salto explosivo real en la tasa de invasión biológica.
- Desagregación por Vías de Introducción (Pathways): Actualmente existe una curación incompleta en los conjuntos de datos nacionales respecto a los vectores que facilitan la entrada de las EEI. La Meta 6 exige identificar y gestionar estas vías de introducción (p. ej., agua de lastre, comercio de mascotas, acuicultura). Se requieren esfuerzos institucionales continuos para armonizar esta información taxonómica y logística, lo cual permitirá transitar desde una métrica de conteo hacia enfoques de modelación refinados que incluyan estimaciones de incertidumbre y desagregación por vías de entrada.